# kubescan — full reproduce pipeline (Colab GPU)

Runs the same pipeline as `make reproduce` (see repo `Makefile`), unmodified — this notebook only
clones the repo, installs dependencies, and shells out to the existing scripts in `research/`.
No training code lives in this notebook; it stays the single source of truth in the repo.

**Before running:** `Runtime > Change runtime type > GPU` (A100/V100/T4 depending on your Colab tier).

In [ ]:
BRANCH = "feat/completing-thesis-and-comments"  # change if the pipeline has since merged

!git clone --branch $BRANCH https://github.com/ObedRav/kubescan.git
%cd kubescan

In [ ]:
# torch itself is left alone — Colab already ships a CUDA-matched build.
# torch-geometric has no compiled-extension dependency in this codebase
# (no torch_scatter/torch_sparse/torch_cluster imports), so a plain pip
# install against Colab's existing torch is sufficient.
!pip install -q torch-geometric scikit-learn skops networkx
!pip install -q -e kubescan/  # local package — augment_graphs.py etc. import from it

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU detected — set Runtime > Change runtime type > GPU"
print("CUDA device:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)

In [ ]:
# ── Run configuration ────────────────────────────────────────────────────
# One notebook covers the standard reproduce run and its ablation arms:
#   FOCAL_LOSS  — False = inverse-frequency weighted CrossEntropy, True = focal loss
#   SELECT_BY   — checkpoint selection metric: "f1" (historical) or "p5" (primary metric)
#   MRR_WEIGHT  — GA chain-MRR tie-breaker; 0.0 (default) = legacy objective
#   RUN_TAG     — results are persisted under this name so arms never clobber each other
SEED       = 42
FOCAL_LOSS = False  # focal loss costs 0.12 CV P@5 vs weighted CE (audit/gnn_p5_refit_2026-07-14.md)
FOCAL_GAMMA = 2.0
SELECT_BY  = "p5"  # CE + P@5 selection is the only config meeting the >0.70 CV P@5 target
MRR_WEIGHT = 0.0  # harmful on the template-heavy OOF pool; see run_ga_ensemble.py Fix 9 note
RUN_TAG    = f"{'focal' if FOCAL_LOSS else 'ce'}_{SELECT_BY}_seed{SEED}"

FOCAL_FLAGS = f"--focal-loss --focal-gamma {FOCAL_GAMMA}" if FOCAL_LOSS else ""
print(f"Run tag: {RUN_TAG}")

In [ ]:
# Optional but recommended: Colab runtimes are ephemeral, so mount Drive
# to persist checkpoints/results after the session ends.
from google.colab import drive

drive.mount("/content/drive")
DRIVE_OUT = "/content/drive/MyDrive/kubescan_checkpoints"
!mkdir -p "$DRIVE_OUT"

## Regenerate data: augmented graphs, cache, and splits

`research/data/graphs/*_aug_*.npz` and `graphs_cache.npz` are gitignored (large, regenerable —
see `.gitignore`); the original cluster graphs and `graph_manifest.csv` **are** tracked, so
augmentation is fully reproducible with a fixed seed. Splits are regenerated here too
(mirrors `make data`): `create_splits.py` is deterministic for a given manifest + seed, and
since the family-aware grouping fix it is the single source of truth for leakage-safe
partitions — template families (badpods_*, datadog_*, …) stay atomic across train/val/test
and across CV folds.

In [ ]:
!python research/scripts/03_augment/augment_graphs.py --seed $SEED
!python research/scripts/04_build_datasets/build_graph_cache.py
!python research/scripts/05_split/create_splits.py --seed $SEED

## Train — RF → GNN (5-fold CV, GPU) → GA ensemble → test evaluation

Mirrors `make reproduce` minus the `data` step (done above) and the redundant fixed-split GNN
pass (removed from the Makefile — `gnn_fold_*.pt` from the CV loop is all downstream steps use).

In [ ]:
!python research/models/train_rf.py --seed $SEED

In [ ]:
# The GPU-bound step. resolve_device() auto-detects CUDA, and
# dataloader_kwargs() enables num_workers/pin_memory only on CUDA
# (on MPS/CPU workers regress performance on in-memory PyG datasets).
%cd research/models
!python train_gnn.py --cv-folds 5 --epochs 300 --hidden 64 --heads 4 --layers 3 \
    $FOCAL_FLAGS --select-by $SELECT_BY --seed $SEED
%cd /content/kubescan

In [ ]:
# GA ensemble weights on out-of-fold predictions. --mrr-weight adds the
# chain-MRR tie-breaker (Fix 9): P@5 plateaus made the GA's pick arbitrary.
%cd research/models
!python run_ga_ensemble.py --oof --mrr-weight $MRR_WEIGHT --seed $SEED
%cd /content/kubescan

In [ ]:
%cd research/models
!python evaluate_test_set.py --show-rankings
%cd /content/kubescan

In [ ]:
!python research/scripts/snapshot_run_manifest.py

## Persist results

Copies checkpoints + results JSON to the mounted Drive folder, and offers a zip download as a
fallback if Drive wasn't mounted.

In [ ]:
import os

if os.path.isdir("/content/drive/MyDrive"):
    RUN_OUT = f"{DRIVE_OUT}/{RUN_TAG}"
    !mkdir -p "$RUN_OUT"
    !cp -r research/models/checkpoints/* "$RUN_OUT/"
    print(f"Copied checkpoints to {RUN_OUT}")
else:
    !zip -r kubescan_checkpoints_$RUN_TAG.zip research/models/checkpoints
    from google.colab import files
    files.download(f"kubescan_checkpoints_{RUN_TAG}.zip")